# ASCENT-G Phase 0 Pilot: HumanEval × Qwen2.5-1.5B-Instruct

**Phase:** 0 — Pipeline validation only  
**Model:** `Qwen2.5-1.5B-Instruct` (registered primary)  
**Task:** HumanEval  
**Method:** GRPO via TRL  

**Acceptance criteria (from execution-plan.md):**
1. Model loads successfully
2. GRPO training runs to planned stopping point without environment failure
3. Adapter weights saved and reloadable
4. Non-degenerate update vector extractable
5. Update vector consumable by analysis code
6. At least one SVD diagnostic runs successfully

This notebook is **not** a benchmark run. Do not interpret reward/accuracy numbers as H1a/H1b/H2 evidence.

## 1. Environment smoke test

In [ ]:
import subprocess, sys, torch

# Capture pre-installed torch version to use as hard pin.
torch_ver = torch.__version__          # e.g. "2.10.0+cu128"
torch_base = torch_ver.split("+")[0]  # strip CUDA suffix for pip constraint

# 1. Upgrade torchao without touching torch (needed by peft >= 0.18).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchao>=0.16.0", "--no-deps"], check=True)

# 2. Install ML packages; constrain torch family to the pre-installed versions.
constraints = (
    f"torch=={torch_base}\n"
    "torchvision\n"
    "torchaudio\n"
    "torchao\n"
).encode()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "trl==0.17.0", "transformers==4.51.3",
                "peft==0.14.0", "accelerate", "datasets",
                "--constraint", "/dev/stdin"],
               input=constraints, check=True)

# Re-import after installs to pick up new versions.
import importlib
for mod in ["trl", "transformers", "peft", "datasets"]:
    importlib.invalidate_caches()

import trl, transformers, peft, datasets
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — abort"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU model  : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"bf16       : {'supported' if bf16_supported else 'NOT supported — will use float16'}")
print(f"torch      : {torch.__version__}")

# Fail fast if assigned GPU is incompatible with torch 2.10+cu128 (sm_60 not supported).
# P100 (sm_60) will crash at LoRA dtype cast regardless of workarounds.
# Request T4 via Kaggle session settings before running.
REQUIRED_GPU = "T4"
if REQUIRED_GPU not in gpu_name:
    raise RuntimeError(
        f"Wrong GPU: got '{gpu_name}', need {REQUIRED_GPU}. "
        f"Go to Session options → Accelerator → GPU T4 x2 and re-run."
    )

HW_META = {
    "gpu_model": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "bf16_supported": bf16_supported,
    "torch_version": torch.__version__,
}
print("\nHW_META:", HW_META)

## 2. Model load

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
MODEL_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    device_map="auto",
)
print("Model loaded:", MODEL_ID)
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 3. LoRA configuration (registered)

In [ ]:
from peft import LoraConfig, get_peft_model

# Registered adapter targets (preregistration v1.3)
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# autocast_adapter_dtype=False: skip dtype cast that fails on P100 with
# torch 2.10+cu128 (no kernel image for SM60).
model = get_peft_model(model, lora_config, autocast_adapter_dtype=False)
model.print_trainable_parameters()

## 4. Load HumanEval dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/openai_humaneval")
train_data = dataset["test"]
print(f"Train size: {len(train_data)}")
print(train_data[0])

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a coding assistant. "
    "Write correct Python code and make the final answer a runnable code block."
)

def format_example(example):
    prompt = str(example["prompt"])
    answer = str(example["canonical_solution"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "answer": answer,
        "raw_prompt": example["prompt"],
        "test": example["test"],
        "entry_point": example["entry_point"],
    }

train_data = train_data.map(format_example)
print(train_data[0]["prompt"][:300])


## 5. Reward function

In [ ]:
def extract_code_block(text):
    """Extract the last fenced code block if present."""
    matches = re.findall(r"```(?:python)?\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if matches:
        return matches[-1]
    return text


def normalize_code_text(text):
    code = extract_code_block(text)
    lines = []
    for raw_line in code.splitlines():
        stripped = raw_line.rstrip()
        if not stripped.strip() or stripped.strip().startswith("#"):
            continue
        lines.append(stripped)
    return "\n".join(lines)


def resolve_humaneval_candidate(raw_prompt, completion, entry_point):
    candidate_code = normalize_code_text(completion)
    if not candidate_code:
        return ""
    if f"def {entry_point}(" in candidate_code:
        return candidate_code
    return str(raw_prompt) + candidate_code


def run_humaneval_check(candidate_program, test_code, entry_point):
    namespace = {}
    exec(candidate_program, namespace)
    exec(str(test_code), namespace)

    check_fn = namespace.get("check")
    if not callable(check_fn):
        raise NameError("HumanEval test block did not define callable check")
    if entry_point not in namespace:
        raise NameError(f"Entry point {entry_point} not found")
    check_fn(namespace[entry_point])
    return True


def humaneval_test_pass(completions, answer, raw_prompt, test, entry_point, **kwargs):
    n_prompts = len(answer)
    assert len(completions) % n_prompts == 0, (
        f"len(completions)={len(completions)} not divisible by n_prompts={n_prompts}"
    )
    n_gen = len(completions) // n_prompts
    rewards = []
    for i, completion in enumerate(completions):
        prompt_idx = i // n_gen
        candidate_program = resolve_humaneval_candidate(
            raw_prompt[prompt_idx],
            completion,
            str(entry_point[prompt_idx]).strip(),
        )
        if not candidate_program:
            rewards.append(0.0)
            continue
        try:
            run_humaneval_check(
                candidate_program,
                test[prompt_idx],
                str(entry_point[prompt_idx]).strip(),
            )
            rewards.append(1.0)
        except Exception:
            rewards.append(0.0)
    return rewards


## 6. GRPO training (pilot — short run)

In [ ]:
import time
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="/kaggle/working/humaneval-qwen2.5-1.5b-phase0",
    max_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    learning_rate=1e-4,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    report_to="none",
)

# `tokenizer` was renamed to `processing_class` in TRL 0.12+.
trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=train_data,
    reward_funcs=humaneval_test_pass,
    processing_class=tokenizer,
)

t0 = time.time()
trainer.train()
total_time = time.time() - t0
step_time_sec = total_time / grpo_config.max_steps

print(f"Training complete.")
print(f"Total time    : {total_time:.1f}s")
print(f"Per-step time : {step_time_sec:.2f}s  (on {HW_META['gpu_model']})")

## 7. Save adapter checkpoint

In [ ]:
ADAPTER_PATH = "/kaggle/working/humaneval-qwen2.5-1.5b-phase0/adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved: {ADAPTER_PATH}")

# Reload with same autocast_adapter_dtype=False as training path to ensure
# the dtype cast CUDA path is not triggered on this hardware/torch combination.
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=MODEL_DTYPE, device_map="auto")
loaded = PeftModel.from_pretrained(base, ADAPTER_PATH, autocast_adapter_dtype=False)
print("Adapter reload: OK")

## 8. Update vector extraction

In [ ]:
import numpy as np
import hashlib, json

REGISTERED_TARGETS = set(LORA_TARGET_MODULES)

def extract_registered_update_vector(peft_model):
    """
    Extract the registered ASCENT-G update object:
        concat(Delta W_A, Delta W_B)
    over all registered LoRA layers in deterministic named_modules() order.

    Returns:
        vector       : 1-D float32 ndarray
        layer_meta   : ordered list of per-layer provenance dicts
    """
    deltas = []
    layer_meta = []
    seen_targets = set()

    for name, module in peft_model.named_modules():
        if not (hasattr(module, "lora_A") and hasattr(module, "lora_B")):
            continue

        A = module.lora_A["default"].weight.detach().float().cpu().numpy()
        B = module.lora_B["default"].weight.detach().float().cpu().numpy()

        deltas.append(A.flatten())
        deltas.append(B.flatten())
        layer_meta.append({
            "name": name,
            "a_shape": list(A.shape),
            "b_shape": list(B.shape),
            "a_numel": int(A.size),
            "b_numel": int(B.size),
            "a_norm": float(np.linalg.norm(A)),
            "b_norm": float(np.linalg.norm(B)),
        })

        for target in REGISTERED_TARGETS:
            if name.endswith(target):
                seen_targets.add(target)

    if not deltas:
        raise ValueError("No LoRA layers found — check adapter config")

    missing = REGISTERED_TARGETS - seen_targets
    if missing:
        raise ValueError(f"Missing registered target modules: {missing}")

    vector = np.concatenate(deltas)
    return vector, layer_meta


update_vector, layer_meta = extract_registered_update_vector(loaded)

vector_bytes = update_vector.tobytes()
checksum = hashlib.sha256(vector_bytes).hexdigest()

print(f"Registered update vector shape : {update_vector.shape}")
print(f"Norm                          : {np.linalg.norm(update_vector):.4f}")
print(f"Non-zero ratio                : {(update_vector != 0).mean():.4f}")
print(f"SHA-256                       : {checksum}")
print(f"Layers captured               : {len(layer_meta)}")

VECTOR_PATH = "/kaggle/working/humaneval-qwen2.5-1.5b-phase0/update_vector.npy"
np.save(VECTOR_PATH, update_vector)

PROVENANCE_PATH = VECTOR_PATH.replace(".npy", "_provenance.json")
provenance = {
    "vector_path": VECTOR_PATH,
    "object_type": "registered_concat_lora_A_B",
    "sha256": checksum,
    "shape": list(update_vector.shape),
    "norm": float(np.linalg.norm(update_vector)),
    "registered_targets": sorted(REGISTERED_TARGETS),
    "layers": layer_meta,
}
with open(PROVENANCE_PATH, "w") as f:
    json.dump(provenance, f, indent=2)

print(f"\nRegistered update vector saved : {VECTOR_PATH}")
print(f"Provenance saved               : {PROVENANCE_PATH}")

## 9. SVD analysis — pilot diagnostic only

In [ ]:
def compute_r90(singular_values):
    """Number of singular values capturing 90% of total variance."""
    total = (singular_values ** 2).sum()
    cumvar = np.cumsum(singular_values ** 2) / total
    return int(np.searchsorted(cumvar, 0.90)) + 1

# Pilot-only diagnostic on dense effective layer deltas.
# This is separate from the registered multi-task H1a/H1b analysis,
# which operates on normalized concat(Delta W_A, Delta W_B) task vectors.
svd_deltas = []
for name, module in loaded.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        scaling = module.scaling["default"]
        delta = (scaling * (B @ A)).detach().float().cpu().numpy()
        svd_deltas.append((name, delta))

results = []
for name, delta in svd_deltas:
    U, S, Vh = np.linalg.svd(delta, full_matrices=False)
    r90 = compute_r90(S)
    results.append({"layer": name, "rank": delta.shape[1], "r90": r90, "s_max": float(S[0])})
    print(f"{name}: r90={r90}, s_max={S[0]:.4f}")

print("\nPilot SVD diagnostic: OK")

## 10. Run report

In [ ]:
import json, datetime

report = {
    "date": datetime.datetime.utcnow().isoformat(),
    "phase": 0,
    "model": MODEL_ID,
    "task": "HumanEval",
    "method": "GRPO",
    "hardware": HW_META,
    "training": {
        "max_steps": grpo_config.max_steps,
        "total_time_sec": round(total_time, 1),
        "per_step_time_sec": round(step_time_sec, 2),
        "precision": "bf16" if bf16_supported else "fp16",
        "phase0_pilot_override": {
            "max_steps": grpo_config.max_steps,
        },
    },
    "update_vector": {
        "object_type": "registered_concat_lora_A_B",
        "shape": list(update_vector.shape),
        "norm": float(np.linalg.norm(update_vector)),
        "sha256": checksum,
        "provenance_path": PROVENANCE_PATH,
        "layers_captured": len(layer_meta),
        "registered_targets": sorted(REGISTERED_TARGETS),
    },
    "svd_results": results,
    "analysis_scope": {
        "registered_primary_analysis_run": False,
        "pilot_svd_diagnostic_run": True,
    },
    "acceptance_criteria": {
        "model_loaded": True,
        "training_completed": True,
        "adapter_saved_and_reloaded": True,
        "update_vector_extracted": True,
        "update_vector_non_degenerate": bool(np.linalg.norm(update_vector) > 0),
        "registered_targets_all_covered": len(REGISTERED_TARGETS - {m["name"].split(".")[-1] for m in layer_meta}) == 0,
        "svd_diagnostic_ran": True,
    },
    "notes": "Fill in: what worked, what failed, what to fix next.",
}

REPORT_PATH = "/kaggle/working/humaneval-qwen2.5-1.5b-phase0/run_report.json"
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nPhase 0 pilot complete.")